# Integração da Base de Alunos com o Censo Escolar 2024

Este notebook dá continuidade ao processo de enriquecimento da base analítica de alunos, incorporando informações do **Censo Escolar da Educação Básica 2024**, disponibilizado pelo **Instituto Nacional de Estudos e Pesquisas Educacionais Anísio Teixeira (INEP)**.

Diferentemente da integração realizada com o Atlas do Desenvolvimento Humano, cujos indicadores já se encontram em granularidade municipal, os microdados públicos do Censo Escolar são disponibilizados no nível dos estabelecimentos de ensino. Como o identificador de escola presente na base de alunos utiliza códigos fictícios, não é possível realizar uma integração direta entre alunos e escolas do Censo.

Por esse motivo, serão construídos **indicadores municipais derivados das características das escolas com matrículas no 2º ano do Ensino Fundamental**, permitindo compatibilizar a granularidade do Censo Escolar com a chave municipal disponível na base de alunos.

A análise será concentrada em três dimensões do contexto educacional:

- acesso à internet para uso nos processos de ensino e aprendizagem;
- oferta de alimentação escolar;
- existência e utilização de biblioteca e/ou sala de leitura.

Os indicadores serão construídos considerando a distribuição das matrículas do 2º ano entre as escolas de cada município, preservando a interpretação contextual das informações e evitando atribuir aos alunos características de escolas que não podem ser diretamente identificadas.

O processo será conduzido de forma controlada, com auditoria prévia dos microdados, definição do universo escolar relevante, construção e validação dos indicadores municipais e, somente depois, integração com a base de alunos enriquecida pelo Atlas.

### Fonte dos dados

Os microdados utilizados neste notebook foram obtidos diretamente no portal oficial do **Instituto Nacional de Estudos e Pesquisas Educacionais Anísio Teixeira (INEP)**:

[Censo Escolar — Microdados](https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/censo-escolar)

A interpretação das variáveis e a definição dos indicadores foram orientadas pelo **Dicionário de Dados da Educação Básica 2024** e pelo **Manual do Usuário dos Microdados do Censo Escolar 2024**, disponibilizados juntamente com os microdados.

## 1. Carregamento das bases

Nesta etapa, serão carregadas as bases necessárias para a construção dos indicadores educacionais e sua posterior integração à base analítica de alunos.

Serão utilizadas:

- **`alunos_atlas.csv`**: base de alunos já enriquecida com os indicadores socioeconômicos do Atlas do Desenvolvimento Humano;
- **`microdados_ed_basica_2024.csv`**: microdados do Censo Escolar da Educação Básica 2024, utilizados para a construção dos indicadores municipais de contexto educacional.

Neste momento, nenhuma transformação será aplicada aos dados. O objetivo é apenas realizar o carregamento das fontes que serão auditadas nas etapas seguintes.

In [0]:
# Objetivo:
# Carregar a base de alunos enriquecida com os indicadores do Atlas
# e os microdados do Censo Escolar da Educação Básica 2024.

# Justificativa:
# As duas bases constituem as fontes necessárias para a construção dos
# indicadores educacionais municipais e sua posterior integração à base de alunos.

# Ação:
# Importar o pandas e carregar os arquivos CSV.

import pandas as pd

df_alunos_atlas = pd.read_csv(
    "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/gold/alunos_atlas/alunos_atlas.csv",
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

df_censo = pd.read_csv(
    "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/gold/fontes_externas/censo_escolar/microdados_ed_basica_2024.csv",
    sep=";",
    encoding="latin-1",
    low_memory=False
)

In [0]:
# Objetivo:
#
# Confirmar o carregamento das bases utilizadas
# no processo de integração.
#
# Justificativa:
#
# A verificação das dimensões permite registrar
# o volume inicial de dados de cada fonte e
# identificar eventuais problemas de leitura
# antes do início da auditoria.
#
# Ação:
#
# Exibe a quantidade de linhas e colunas
# presentes em cada DataFrame.

print("Base de alunos + Atlas:", df_alunos_atlas.shape)
print("Base Censo Escolar 2024:", df_censo.shape)

## 2. Auditoria da base do Censo Escolar

Nesta etapa será analisada a estrutura da base do Censo Escolar da
Educação Básica 2024, com atenção à sua granularidade, situação de
funcionamento dos estabelecimentos, cobertura municipal e consistência
dos dados.

A auditoria fornecerá os elementos necessários para definir de forma
segura o universo de estabelecimentos utilizado na construção dos
indicadores municipais e sua posterior integração com a base de alunos.

In [0]:
# Objetivo:
#
# Analisar a situação de funcionamento dos
# estabelecimentos presentes no Censo Escolar 2024.
#
# Justificativa:
#
# Segundo o Dicionário de Dados do Censo Escolar 2024,
# a variável TP_SITUACAO_FUNCIONAMENTO representa a
# situação de funcionamento do estabelecimento de ensino.
#
# O dicionário oficial estabelece a seguinte codificação:
#
# 1 = Em Atividade
# 2 = Paralisada
# 3 = Extinta (ano do Censo)
# 4 = Extinta em Anos Anteriores
#
# A identificação dessas categorias permite compreender
# a composição da base antes da definição do universo
# de estabelecimentos que será utilizado na construção
# dos indicadores educacionais.
#
# Ação:
#
# Define, conforme o dicionário oficial, o mapeamento
# entre os códigos e seus respectivos significados e
# contabiliza os estabelecimentos em cada situação de
# funcionamento.

situacao_funcionamento = {
    1: "Em Atividade",
    2: "Paralisada",
    3: "Extinta (ano do Censo)",
    4: "Extinta em Anos Anteriores"
}

(
    df_censo["TP_SITUACAO_FUNCIONAMENTO"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("codigo")
    .to_frame("quantidade")
    .assign(
        situacao=lambda x: x.index.map(situacao_funcionamento)
    )
)

### Definição do universo de estabelecimentos

A auditoria da variável `TP_SITUACAO_FUNCIONAMENTO` mostrou que os
215.545 estabelecimentos presentes nos microdados do Censo Escolar
2024 estão distribuídos entre três situações de funcionamento:

- **181.065** estabelecimentos em atividade;
- **31.321** estabelecimentos paralisados;
- **3.159** estabelecimentos extintos no ano do Censo.

Embora o dicionário oficial também preveja o código `4` para
estabelecimentos extintos em anos anteriores, essa categoria não foi
observada na base analisada.

Como os indicadores educacionais serão construídos para representar
as condições associadas às matrículas do 2º ano do Ensino Fundamental,
serão considerados somente os estabelecimentos classificados como
**em atividade** (`TP_SITUACAO_FUNCIONAMENTO = 1`).

Esse recorte evita incorporar ao cálculo estabelecimentos paralisados
ou extintos, que não representam unidades escolares em funcionamento
no período analisado.

In [0]:
# Objetivo:
#
# Delimitar os estabelecimentos do Censo Escolar 2024
# que estavam em atividade.
#
# Justificativa:
#
# A auditoria da variável TP_SITUACAO_FUNCIONAMENTO
# mostrou que a base contém estabelecimentos em atividade,
# paralisados e extintos no ano do Censo.
#
# Conforme o Dicionário de Dados do Censo Escolar 2024,
# o código 1 da variável TP_SITUACAO_FUNCIONAMENTO
# corresponde à categoria "Em Atividade".
#
# Como os indicadores educacionais devem representar
# estabelecimentos efetivamente em funcionamento no
# período analisado, somente os registros dessa categoria
# serão considerados nas próximas etapas.
#
# Ação:
#
# Seleciona os estabelecimentos classificados como
# "Em Atividade" e cria uma cópia para preservar
# a base original do Censo Escolar.

df_censo_ativo = df_censo[
    df_censo["TP_SITUACAO_FUNCIONAMENTO"] == 1
].copy()

df_censo_ativo.shape

In [0]:
# Objetivo:
#
# Analisar a variável QT_MAT_FUND_AI_2 nos
# estabelecimentos em atividade do Censo Escolar 2024.
#
# Justificativa:
#
# Segundo o Dicionário de Dados do Censo Escolar 2024,
# a variável QT_MAT_FUND_AI_2 representa o número de
# matrículas do Ensino Fundamental - Anos Iniciais -
# 2º Ano.
#
# Essa variável é fundamental para identificar quais
# estabelecimentos em atividade possuem matrículas no
# 2º ano do Ensino Fundamental, etapa de ensino
# correspondente ao universo educacional de interesse
# deste projeto.
#
# Antes de utilizá-la como critério para delimitar esse
# universo, é necessário avaliar sua completude e a
# distribuição dos valores observados.
#
# Ação:
#
# Resume a quantidade de valores ausentes, iguais a zero
# e maiores que zero da variável QT_MAT_FUND_AI_2 entre
# os estabelecimentos em atividade.

pd.Series({
    "ausentes": df_censo_ativo["QT_MAT_FUND_AI_2"].isna().sum(),
    "iguais_a_zero": df_censo_ativo["QT_MAT_FUND_AI_2"].eq(0).sum(),
    "maiores_que_zero": df_censo_ativo["QT_MAT_FUND_AI_2"].gt(0).sum()
})

### Delimitação das escolas com matrículas no 2º ano

Após a seleção dos estabelecimentos em atividade, foi analisada a
variável `QT_MAT_FUND_AI_2`, que representa o número de matrículas
do Ensino Fundamental - Anos Iniciais - 2º Ano.

Entre os 181.065 estabelecimentos em atividade:

- **1.779** apresentam valor ausente para a variável;
- **84.331** apresentam zero matrículas no 2º ano;
- **94.955** apresentam pelo menos uma matrícula no 2º ano.

Os valores ausentes foram mantidos conceitualmente separados dos
valores iguais a zero, uma vez que ausência de informação não equivale
à inexistência de matrículas.

Como os indicadores municipais serão construídos para representar as
condições educacionais associadas às matrículas do 2º ano, serão
considerados somente os estabelecimentos com
`QT_MAT_FUND_AI_2 > 0`.

Esse critério delimita o universo às escolas que efetivamente possuem
matrículas na etapa de ensino de interesse do projeto, sem atribuir
peso a estabelecimentos sem matrículas registradas no 2º ano ou com
informação ausente para essa variável.

In [0]:
# Objetivo:
#
# Delimitar os estabelecimentos em atividade que
# possuem matrículas no 2º ano do Ensino Fundamental.
#
# Justificativa:
#
# Segundo o Dicionário de Dados do Censo Escolar 2024,
# a variável QT_MAT_FUND_AI_2 representa o número de
# matrículas do Ensino Fundamental - Anos Iniciais -
# 2º Ano.
#
# A auditoria mostrou que, entre os estabelecimentos
# em atividade, existem registros com valores ausentes,
# iguais a zero e maiores que zero nessa variável.
#
# Como os indicadores municipais serão construídos para
# representar as condições educacionais associadas às
# matrículas do 2º ano, serão considerados somente os
# estabelecimentos com pelo menos uma matrícula
# registrada nessa etapa de ensino.
#
# Ação:
#
# Seleciona os estabelecimentos em atividade com
# QT_MAT_FUND_AI_2 maior que zero e cria uma cópia
# para preservar as etapas anteriores da auditoria.

df_censo_2ano = df_censo_ativo[
    df_censo_ativo["QT_MAT_FUND_AI_2"] > 0
].copy()

df_censo_2ano.shape

### Seleção das variáveis para construção dos indicadores

Com o objetivo de incorporar informações do contexto educacional
relevantes para a alfabetização sem aumentar excessivamente a
dimensionalidade da base analítica, foram selecionadas quatro
variáveis do Censo Escolar da Educação Básica 2024.

A seleção foi realizada com base nas definições disponíveis no
dicionário oficial da fonte e buscou representar características
das escolas associadas às matrículas do 2º ano do Ensino Fundamental.

Como os microdados possuem granularidade por estabelecimento de
ensino, essas variáveis serão utilizadas posteriormente na construção
de indicadores agregados em nível municipal.

| Variável | Definição | Função analítica | Justificativa |
| --- | --- | --- | --- |
| `QT_MAT_FUND_AI_2` | Número de Matrículas do Ensino Fundamental - Anos Iniciais - 2º Ano | Ponderação | Representar o número de matrículas do 2º ano em cada estabelecimento e permitir que os indicadores municipais sejam ponderados pelo volume de matrículas. |
| `IN_INTERNET_APRENDIZAGEM` | Acesso à Internet - Para uso nos processos de ensino e aprendizagem | Infraestrutura digital | Representar a disponibilidade de acesso à internet destinado aos processos de ensino e aprendizagem. |
| `IN_ALIMENTACAO` | Alimentação escolar para os alunos - PNAE/FNDE | Assistência ao estudante | Representar a oferta de alimentação escolar aos alunos. |
| `IN_BIBLIOTECA_SALA_LEITURA` | Dependências físicas existentes e utilizadas na escola - Biblioteca e/ou Sala de leitura | Infraestrutura pedagógica | Representar a existência e utilização de espaços escolares relacionados à leitura e ao acesso a recursos pedagógicos. |

As três características educacionais possuem codificação binária
no dicionário oficial:

- `IN_INTERNET_APRENDIZAGEM`: **0 = Não** e **1 = Sim**;
- `IN_ALIMENTACAO`: **0 = Não oferece** e **1 = Oferece**;
- `IN_BIBLIOTECA_SALA_LEITURA`: **0 = Não** e **1 = Sim**.

A variável `CO_MUNICIPIO` será mantida adicionalmente como chave
técnica para a agregação dos indicadores e sua posterior integração
com a base de alunos.

As variáveis `TP_DEPENDENCIA` e `TP_LOCALIZACAO` serão mantidas como
variáveis auxiliares de auditoria, permitindo compreender a composição
do universo de estabelecimentos selecionado antes da construção dos
indicadores municipais.

### Auditoria das variáveis selecionadas

Antes da construção dos indicadores municipais, as variáveis
selecionadas serão submetidas a uma auditoria para avaliar sua
estrutura, completude e consistência no universo de estabelecimentos
em atividade com matrículas no 2º ano do Ensino Fundamental.

Essa etapa permitirá verificar se as variáveis apresentam condições
adequadas para serem utilizadas na construção dos indicadores e
identificar eventuais particularidades que precisem ser consideradas
antes da agregação em nível municipal.

In [0]:
# Objetivo:
#
# Avaliar a estrutura e a completude das variáveis
# selecionadas para a construção dos indicadores
# municipais do Censo Escolar 2024.
#
# Justificativa:
#
# Antes da construção dos indicadores, é necessário
# verificar se as variáveis apresentam tipos compatíveis,
# valores ausentes e diversidade suficiente no universo
# de estabelecimentos em atividade com matrículas no
# 2º ano do Ensino Fundamental.
#
# Segundo o Dicionário de Dados do Censo Escolar 2024:
#
# QT_MAT_FUND_AI_2 representa o número de matrículas
# do Ensino Fundamental - Anos Iniciais - 2º Ano.
#
# IN_INTERNET_APRENDIZAGEM representa o acesso à
# internet para uso nos processos de ensino e aprendizagem.
#
# IN_ALIMENTACAO representa a oferta de alimentação
# escolar para os alunos - PNAE/FNDE.
#
# IN_BIBLIOTECA_SALA_LEITURA representa a existência
# e utilização de biblioteca e/ou sala de leitura.
#
# Ação:
#
# Resume o tipo, a quantidade e o percentual de valores
# ausentes e o número de valores distintos observados
# em cada variável selecionada.

variaveis_censo = [
    "QT_MAT_FUND_AI_2",
    "IN_INTERNET_APRENDIZAGEM",
    "IN_ALIMENTACAO",
    "IN_BIBLIOTECA_SALA_LEITURA"
]

auditoria_variaveis_censo = pd.DataFrame({
    "tipo": df_censo_2ano[variaveis_censo].dtypes,
    "ausentes": df_censo_2ano[variaveis_censo].isna().sum(),
    "percentual_ausentes": (
        df_censo_2ano[variaveis_censo]
        .isna()
        .mean()
        .mul(100)
        .round(2)
    ),
    "valores_distintos": (
        df_censo_2ano[variaveis_censo]
        .nunique(dropna=False)
    )
})

auditoria_variaveis_censo

In [0]:
# Objetivo:
#
# Verificar os valores observados nas variáveis binárias
# selecionadas para a construção dos indicadores municipais.
#
# Justificativa:
#
# Segundo o Dicionário de Dados do Censo Escolar 2024,
# as três variáveis possuem domínio binário:
#
# IN_INTERNET_APRENDIZAGEM:
# 0 = Não
# 1 = Sim
#
# IN_ALIMENTACAO:
# 0 = Não oferece
# 1 = Oferece
#
# IN_BIBLIOTECA_SALA_LEITURA:
# 0 = Não
# 1 = Sim
#
# A auditoria anterior mostrou dois valores distintos em
# cada variável, mas essa informação, isoladamente, não
# confirma que os valores observados correspondam aos
# códigos 0 e 1 definidos no dicionário oficial.
#
# Ação:
#
# Contabiliza os registros de cada valor observado nas
# três variáveis binárias selecionadas.

variaveis_binarias = [
    "IN_INTERNET_APRENDIZAGEM",
    "IN_ALIMENTACAO",
    "IN_BIBLIOTECA_SALA_LEITURA"
]

for variavel in variaveis_binarias:
    print(f"\n{variavel}")
    print(
        df_censo_2ano[variavel]
        .value_counts(dropna=False)
        .sort_index()
    )

In [0]:
# Objetivo:
#
# Avaliar a distribuição do número de matrículas
# do 2º ano do Ensino Fundamental nos estabelecimentos
# selecionados para a análise.
#
# Justificativa:
#
# Segundo o Dicionário de Dados do Censo Escolar 2024,
# a variável QT_MAT_FUND_AI_2 representa o número de
# matrículas do Ensino Fundamental - Anos Iniciais -
# 2º Ano.
#
# Essa variável será utilizada como ponderador na
# construção dos indicadores municipais, fazendo com
# que estabelecimentos com maior número de matrículas
# no 2º ano tenham peso proporcionalmente maior.
#
# Por esse motivo, é necessário conhecer sua distribuição
# e verificar a presença de valores extremos antes de
# utilizá-la nos cálculos.
#
# Ação:
#
# Calcula estatísticas descritivas da variável
# QT_MAT_FUND_AI_2 no universo de estabelecimentos
# em atividade com matrículas no 2º ano.

df_censo_2ano[["QT_MAT_FUND_AI_2"]].describe().T

In [0]:
# Objetivo:
#
# Investigar os maiores valores observados na variável
# QT_MAT_FUND_AI_2.
#
# Justificativa:
#
# A análise descritiva mostrou que o número de matrículas
# do 2º ano apresenta distribuição assimétrica à direita,
# com mediana de 17 matrículas, terceiro quartil de 41
# e valor máximo de 458.
#
# Como QT_MAT_FUND_AI_2 será utilizada como ponderador
# na construção dos indicadores municipais, valores
# elevados terão maior influência nos cálculos.
#
# Valores extremos não representam necessariamente
# inconsistências, mas devem ser examinados antes de sua
# utilização para verificar a plausibilidade dos registros.
#
# Ação:
#
# Exibe os 20 maiores valores de QT_MAT_FUND_AI_2
# observados no universo selecionado.

(
    df_censo_2ano[["QT_MAT_FUND_AI_2"]]
    .sort_values(
        "QT_MAT_FUND_AI_2",
        ascending=False
    )
    .head(20)
)

In [0]:
# Objetivo:
#
# Analisar a distribuição dos estabelecimentos
# selecionados segundo a dependência administrativa.
#
# Justificativa:
#
# Segundo o Dicionário de Dados do Censo Escolar 2024,
# a variável TP_DEPENDENCIA representa a Dependência
# Administrativa do estabelecimento de ensino.
#
# O dicionário oficial estabelece a seguinte codificação:
#
# 1 = Federal
# 2 = Estadual
# 3 = Municipal
# 4 = Privada
#
# A análise dessa variável permite compreender a
# composição administrativa do universo de escolas
# em atividade com matrículas no 2º ano antes da
# construção dos indicadores municipais.
#
# Ação:
#
# Define, conforme o dicionário oficial, o mapeamento
# entre os códigos e seus respectivos significados e
# contabiliza os estabelecimentos por dependência
# administrativa.

dependencia_administrativa = {
    1: "Federal",
    2: "Estadual",
    3: "Municipal",
    4: "Privada"
}

(
    df_censo_2ano["TP_DEPENDENCIA"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("codigo")
    .to_frame("quantidade")
    .assign(
        dependencia=lambda x: x.index.map(dependencia_administrativa)
    )
)

In [0]:
# Objetivo:
#
# Analisar a distribuição dos estabelecimentos
# selecionados segundo sua localização.
#
# Justificativa:
#
# Segundo o Dicionário de Dados do Censo Escolar 2024,
# a variável TP_LOCALIZACAO representa a Localização
# do estabelecimento de ensino.
#
# O dicionário oficial estabelece a seguinte codificação:
#
# 1 = Urbana
# 2 = Rural
#
# A análise dessa variável permite compreender a
# distribuição territorial do universo de escolas
# em atividade com matrículas no 2º ano antes da
# construção dos indicadores municipais.
#
# Ação:
#
# Define, conforme o dicionário oficial, o mapeamento
# entre os códigos e seus respectivos significados e
# contabiliza os estabelecimentos por localização.

localizacao = {
    1: "Urbana",
    2: "Rural"
}

(
    df_censo_2ano["TP_LOCALIZACAO"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("codigo")
    .to_frame("quantidade")
    .assign(
        localizacao=lambda x: x.index.map(localizacao)
    )
)

In [0]:
# Objetivo:
#
# Verificar as dependências administrativas presentes
# na base de alunos + Atlas.
#
# Justificativa:
#
# A auditoria do Censo Escolar 2024 mostrou que o universo
# de estabelecimentos com matrículas no 2º ano contempla
# as dependências Federal, Estadual, Municipal e Privada.
#
# Como a base de alunos também possui a variável
# TP_DEPENDENCIA, é necessário verificar quais categorias
# estão efetivamente representadas nessa população antes
# de avaliar a comparabilidade entre as duas fontes.
#
# Ação:
#
# Contabiliza os alunos por código de dependência
# administrativa.

df_alunos_atlas["TP_DEPENDENCIA"].value_counts(
    dropna=False
).sort_index()

In [0]:
# Objetivo:
#
# Analisar a distribuição das matrículas do 2º ano
# segundo a dependência administrativa dos
# estabelecimentos selecionados.
#
# Justificativa:
#
# A auditoria mostrou que a base de alunos contempla
# somente as dependências Estadual e Municipal, enquanto
# o universo selecionado do Censo Escolar também inclui
# estabelecimentos Federais e Privados.
#
# Segundo o Dicionário de Dados do Censo Escolar 2024,
# a variável TP_DEPENDENCIA possui a seguinte codificação:
#
# 1 = Federal
# 2 = Estadual
# 3 = Municipal
# 4 = Privada
#
# Como QT_MAT_FUND_AI_2 será utilizada como ponderador
# na construção dos indicadores municipais, é necessário
# avaliar a participação de cada dependência no total de
# matrículas do 2º ano antes de definir o universo final
# utilizado nos cálculos.
#
# Ação:
#
# Agrupa os estabelecimentos por dependência
# administrativa e calcula o número de escolas e a soma
# das matrículas do 2º ano em cada categoria.

(
    df_censo_2ano
    .groupby("TP_DEPENDENCIA")
    .agg(
        quantidade_escolas=("TP_DEPENDENCIA", "size"),
        matriculas_2ano=("QT_MAT_FUND_AI_2", "sum")
    )
    .assign(
        dependencia=lambda x: x.index.map(dependencia_administrativa),
        percentual_matriculas=lambda x: (
            x["matriculas_2ano"]
            / x["matriculas_2ano"].sum()
            * 100
        ).round(2)
    )
)

### Alinhamento entre o universo do Censo Escolar e a base de alunos

A auditoria da dependência administrativa mostrou que os
estabelecimentos em atividade com matrículas no 2º ano pertencem
às redes Federal, Estadual, Municipal e Privada.

Entretanto, a base de alunos utilizada no projeto contempla somente
as dependências Estadual e Municipal, desconsiderando os 510 registros
sem informação de dependência administrativa.

A análise das matrículas do 2º ano no Censo Escolar mostrou a seguinte
distribuição:

- **0,05%** na rede Federal;
- **10,31%** na rede Estadual;
- **70,19%** na rede Municipal;
- **19,46%** na rede Privada.

As redes Estadual e Municipal concentram, portanto, **80,50% das
matrículas do 2º ano** presentes no universo anteriormente selecionado.

Como os indicadores municipais serão posteriormente associados a
alunos pertencentes exclusivamente às redes Estadual e Municipal,
sua construção será restrita aos estabelecimentos dessas mesmas
dependências administrativas.

Esse alinhamento busca garantir que os indicadores representem o
contexto educacional das redes efetivamente contempladas pela
população de alunos analisada, evitando que características de
estabelecimentos Federais ou Privados influenciem os indicadores
atribuídos a estudantes das redes Estadual e Municipal.

In [0]:
# Objetivo:
#
# Alinhar o universo de estabelecimentos do Censo
# Escolar às dependências administrativas presentes
# na base de alunos.
#
# Justificativa:
#
# A auditoria mostrou que a base de alunos contempla
# somente as dependências Estadual e Municipal, enquanto
# o Censo Escolar também contém estabelecimentos das
# redes Federal e Privada.
#
# Segundo o Dicionário de Dados do Censo Escolar 2024,
# a variável TP_DEPENDENCIA possui a seguinte codificação:
#
# 1 = Federal
# 2 = Estadual
# 3 = Municipal
# 4 = Privada
#
# Como os indicadores municipais serão associados a
# alunos das redes Estadual e Municipal, o universo
# utilizado em sua construção será restrito às mesmas
# dependências administrativas.
#
# Esse recorte evita que características de escolas
# Federais ou Privadas influenciem os indicadores
# atribuídos à população analisada.
#
# Ação:
#
# Seleciona os estabelecimentos das redes Estadual
# e Municipal e cria uma cópia para preservar as
# etapas anteriores da auditoria.

df_censo_2ano_rede = df_censo_2ano[
    df_censo_2ano["TP_DEPENDENCIA"].isin([2, 3])
].copy()

df_censo_2ano_rede.shape

In [0]:
# Objetivo:
#
# Revalidar as variáveis que serão utilizadas na
# construção dos indicadores municipais após a
# definição do universo definitivo de estabelecimentos.
#
# Justificativa:
#
# O universo de análise foi restringido aos
# estabelecimentos em atividade, com matrículas no
# 2º ano do Ensino Fundamental e pertencentes às
# redes Estadual ou Municipal.
#
# Como essa última seleção alterou a população de
# 94.955 para 72.388 estabelecimentos, é necessário
# confirmar que as condições verificadas anteriormente
# permanecem válidas no universo definitivo.
#
# Ação:
#
# Verifica a quantidade de valores ausentes, o menor
# valor de QT_MAT_FUND_AI_2 e os valores distintos
# observados nas três variáveis binárias.

print(
    "Ausentes em QT_MAT_FUND_AI_2:",
    df_censo_2ano_rede["QT_MAT_FUND_AI_2"].isna().sum()
)

print(
    "Menor valor de QT_MAT_FUND_AI_2:",
    df_censo_2ano_rede["QT_MAT_FUND_AI_2"].min()
)

for variavel in variaveis_binarias:
    print(f"\n{variavel}")
    print(
        "Ausentes:",
        df_censo_2ano_rede[variavel].isna().sum()
    )
    print(
        "Valores observados:",
        sorted(df_censo_2ano_rede[variavel].dropna().unique())
    )

### Resultado da auditoria das variáveis selecionadas

A auditoria confirmou a consistência das variáveis selecionadas no
universo definitivo de estabelecimentos utilizado para a construção
dos indicadores municipais.

Esse universo é composto por escolas:

- em atividade;
- com pelo menos uma matrícula registrada no 2º ano do Ensino
  Fundamental;
- pertencentes às redes Estadual ou Municipal, em alinhamento com
  as dependências administrativas presentes na base de alunos.

Após esses recortes, foram mantidos **72.388 estabelecimentos**.

A variável `QT_MAT_FUND_AI_2`, utilizada como ponderador, não apresenta
valores ausentes e possui valor mínimo igual a 1, garantindo que todos
os estabelecimentos selecionados possuem matrículas registradas no
2º ano.

As variáveis `IN_INTERNET_APRENDIZAGEM`, `IN_ALIMENTACAO` e
`IN_BIBLIOTECA_SALA_LEITURA` também não apresentam valores ausentes
e possuem exclusivamente os valores 0 e 1, em conformidade com os
domínios definidos no dicionário oficial.

Dessa forma, não foram identificadas inconsistências que exijam
tratamento adicional dessas variáveis antes da construção dos
indicadores municipais.

## 3. Construção dos indicadores municipais

Após a definição e auditoria do universo de estabelecimentos, serão
construídos indicadores municipais a partir das características
educacionais selecionadas no Censo Escolar 2024.

Como os estabelecimentos possuem diferentes quantidades de matrículas
no 2º ano, os indicadores serão ponderados por `QT_MAT_FUND_AI_2`.
Dessa forma, cada escola contribuirá para o indicador municipal
proporcionalmente ao número de matrículas do 2º ano que possui.

Para uma característica binária \(X_i\), em que 1 indica a presença
da característica e 0 sua ausência, o indicador do município será
calculado por:

\[
\text{Indicador municipal} =
\frac{\sum_{i=1}^{n}(QT\_MAT\_FUND\_AI\_2_i \times X_i)}
{\sum_{i=1}^{n}QT\_MAT\_FUND\_AI\_2_i}
\]

Assim, o numerador representa o número de matrículas do 2º ano
associadas a estabelecimentos que possuem determinada característica,
enquanto o denominador representa o total de matrículas do 2º ano
consideradas no município.

Serão construídos três indicadores:

| Indicador | Numerador | Denominador | Interpretação |
| --- | --- | --- | --- |
| `prop_mat_2ano_internet_aprendizagem` | Matrículas do 2º ano em escolas com internet para uso nos processos de ensino e aprendizagem | Total de matrículas do 2º ano | Proporção das matrículas do 2º ano associadas a escolas com internet destinada aos processos de ensino e aprendizagem. |
| `prop_mat_2ano_alimentacao` | Matrículas do 2º ano em escolas que oferecem alimentação escolar - PNAE/FNDE | Total de matrículas do 2º ano | Proporção das matrículas do 2º ano associadas a escolas que oferecem alimentação escolar. |
| `prop_mat_2ano_biblioteca_sala_leitura` | Matrículas do 2º ano em escolas com biblioteca e/ou sala de leitura | Total de matrículas do 2º ano | Proporção das matrículas do 2º ano associadas a escolas com biblioteca e/ou sala de leitura. |

Os indicadores serão calculados exclusivamente a partir dos
estabelecimentos em atividade, com matrículas no 2º ano e pertencentes
às redes Estadual ou Municipal.

In [0]:
# Objetivo:
#
# Auditar a chave municipal que será utilizada na
# agregação dos indicadores do Censo Escolar 2024.
#
# Justificativa:
#
# Segundo o Dicionário de Dados do Censo Escolar 2024,
# a variável CO_MUNICIPIO representa o Código do Município.
#
# Como os indicadores serão construídos por município e
# posteriormente integrados à base de alunos, é necessário
# verificar a completude, o tipo e a quantidade de
# municípios representados no universo definitivo antes
# da agregação.
#
# Ação:
#
# Verifica o tipo da variável CO_MUNICIPIO, a quantidade
# de valores ausentes e o número de municípios distintos
# presentes no universo definitivo.

pd.Series({
    "tipo": df_censo_2ano_rede["CO_MUNICIPIO"].dtype,
    "ausentes": df_censo_2ano_rede["CO_MUNICIPIO"].isna().sum(),
    "municipios_distintos": df_censo_2ano_rede["CO_MUNICIPIO"].nunique()
})

In [0]:
# Objetivo:
#
# Construir o indicador municipal de proporção das
# matrículas do 2º ano associadas a estabelecimentos
# com internet para uso nos processos de ensino
# e aprendizagem.
#
# Justificativa:
#
# Segundo o Dicionário de Dados do Censo Escolar 2024,
# IN_INTERNET_APRENDIZAGEM indica a existência de acesso
# à internet para uso nos processos de ensino e
# aprendizagem:
#
# 0 = Não
# 1 = Sim
#
# Como os estabelecimentos possuem diferentes quantidades
# de matrículas no 2º ano, o indicador será ponderado por
# QT_MAT_FUND_AI_2.
#
# O numerador corresponde às matrículas do 2º ano
# associadas a estabelecimentos com a característica,
# enquanto o denominador corresponde ao total de
# matrículas do 2º ano no município.
#
# Ação:
#
# Calcula, para cada estabelecimento, a quantidade de
# matrículas do 2º ano associadas à presença de internet
# para aprendizagem.

df_censo_2ano_rede["mat_2ano_internet_aprendizagem"] = (
    df_censo_2ano_rede["QT_MAT_FUND_AI_2"]
    * df_censo_2ano_rede["IN_INTERNET_APRENDIZAGEM"]
)

df_censo_2ano_rede[
    [
        "CO_MUNICIPIO",
        "QT_MAT_FUND_AI_2",
        "IN_INTERNET_APRENDIZAGEM",
        "mat_2ano_internet_aprendizagem"
    ]
].head(10)

In [0]:
# Objetivo:
#
# Agregar por município as matrículas do 2º ano e
# as matrículas associadas a estabelecimentos com
# internet para uso nos processos de ensino e
# aprendizagem.
#
# Justificativa:
#
# A etapa anterior calculou, para cada estabelecimento,
# a quantidade de matrículas do 2º ano associadas à
# presença de internet para aprendizagem.
#
# Para construir o indicador municipal, é necessário
# somar, dentro de cada município, tanto essas matrículas
# quanto o total de matrículas do 2º ano.
#
# A razão entre essas duas quantidades representará a
# proporção ponderada de matrículas associadas a escolas
# com internet para aprendizagem.
#
# Ação:
#
# Agrupa os estabelecimentos por CO_MUNICIPIO e calcula
# o total de matrículas do 2º ano e o total de matrículas
# associadas à presença de internet para aprendizagem.

df_indicadores_censo = (
    df_censo_2ano_rede
    .groupby("CO_MUNICIPIO", as_index=False)
    .agg(
        total_mat_2ano=("QT_MAT_FUND_AI_2", "sum"),
        mat_2ano_internet_aprendizagem=(
            "mat_2ano_internet_aprendizagem",
            "sum"
        )
    )
)

df_indicadores_censo.head()

In [0]:
# Objetivo:
#
# Validar a granularidade da base municipal produzida
# pela agregação dos estabelecimentos do Censo Escolar.
#
# Justificativa:
#
# Antes da agregação, foram identificados 5.570 códigos
# municipais distintos no universo definitivo.
#
# Como o agrupamento foi realizado por CO_MUNICIPIO,
# espera-se obter exatamente um registro por município
# e nenhuma duplicidade nessa chave.
#
# Ação:
#
# Verifica a dimensão da base agregada, a quantidade
# de municípios distintos e a existência de códigos
# municipais duplicados.

pd.Series({
    "registros": len(df_indicadores_censo),
    "municipios_distintos": (
        df_indicadores_censo["CO_MUNICIPIO"].nunique()
    ),
    "municipios_duplicados": (
        df_indicadores_censo["CO_MUNICIPIO"].duplicated().sum()
    )
})

In [0]:
# Objetivo:
#
# Calcular a proporção municipal de matrículas do
# 2º ano associadas a estabelecimentos com internet
# para uso nos processos de ensino e aprendizagem.
#
# Justificativa:
#
# A agregação municipal produziu uma base com uma linha
# por município, contendo o total de matrículas do 2º ano
# e o total dessas matrículas associadas a estabelecimentos
# com internet para aprendizagem.
#
# A auditoria confirmou a presença de 5.570 municípios
# distintos e ausência de duplicidades na chave
# CO_MUNICIPIO.
#
# O indicador é obtido pela razão entre as matrículas
# associadas a estabelecimentos com internet para
# aprendizagem e o total de matrículas do 2º ano
# consideradas em cada município.
#
# Ação:
#
# Calcula o indicador
# prop_mat_2ano_internet_aprendizagem para cada município.

df_indicadores_censo["prop_mat_2ano_internet_aprendizagem"] = (
    df_indicadores_censo["mat_2ano_internet_aprendizagem"]
    / df_indicadores_censo["total_mat_2ano"]
)

df_indicadores_censo.head()

In [0]:
# Objetivo:
#
# Validar o indicador municipal de proporção das
# matrículas do 2º ano associadas a estabelecimentos
# com internet para aprendizagem.
#
# Justificativa:
#
# Por definição, uma proporção deve assumir valores
# entre 0 e 1.
#
# Além disso, como todos os municípios possuem matrículas
# do 2º ano no universo selecionado, não são esperados
# valores ausentes decorrentes de denominadores nulos.
#
# A verificação desses limites permite confirmar a
# consistência matemática do indicador construído.
#
# Ação:
#
# Verifica a quantidade de valores ausentes, o menor
# e o maior valor observados e a existência de valores
# fora do intervalo esperado entre 0 e 1.

indicador_internet = (
    df_indicadores_censo["prop_mat_2ano_internet_aprendizagem"]
)

pd.Series({
    "ausentes": indicador_internet.isna().sum(),
    "minimo": indicador_internet.min(),
    "maximo": indicador_internet.max(),
    "fora_intervalo": (
        ~indicador_internet.between(0, 1)
    ).sum()
})

### Validação do primeiro indicador

O indicador `prop_mat_2ano_internet_aprendizagem` foi construído
inicialmente de forma isolada para validar a metodologia de ponderação
adotada.

O cálculo foi realizado a partir da razão entre o número de matrículas
do 2º ano associadas a estabelecimentos com internet para uso nos
processos de ensino e aprendizagem e o total de matrículas do 2º ano
consideradas em cada município.

A agregação resultou em **5.570 registros municipais**, sem
duplicidades na chave `CO_MUNICIPIO`.

A validação do indicador confirmou:

- ausência de valores ausentes;
- valor mínimo igual a **0**;
- valor máximo igual a **1**;
- ausência de valores fora do intervalo esperado `[0, 1]`.

Esses resultados confirmam a consistência matemática do procedimento
utilizado para a construção da proporção ponderada.

A mesma metodologia será aplicada aos indicadores de alimentação
escolar e biblioteca e/ou sala de leitura.

In [0]:
# Objetivo:
#
# Construir os indicadores municipais de proporção das
# matrículas do 2º ano associadas a estabelecimentos
# que oferecem alimentação escolar e que possuem
# biblioteca e/ou sala de leitura.
#
# Justificativa:
#
# A metodologia de ponderação por QT_MAT_FUND_AI_2 foi
# previamente aplicada e validada na construção do
# indicador de internet para aprendizagem.
#
# Segundo o Dicionário de Dados do Censo Escolar 2024:
#
# IN_ALIMENTACAO:
# 0 = Não oferece
# 1 = Oferece
#
# IN_BIBLIOTECA_SALA_LEITURA:
# 0 = Não
# 1 = Sim
#
# Como essas variáveis possuem codificação binária, sua
# multiplicação por QT_MAT_FUND_AI_2 permite identificar
# quantas matrículas do 2º ano estão associadas a
# estabelecimentos que apresentam cada característica.
#
# Ação:
#
# Calcula os numeradores por estabelecimento, agrega-os
# por município e divide cada resultado pelo total de
# matrículas do 2º ano já calculado anteriormente.

df_censo_2ano_rede["mat_2ano_alimentacao"] = (
    df_censo_2ano_rede["QT_MAT_FUND_AI_2"]
    * df_censo_2ano_rede["IN_ALIMENTACAO"]
)

df_censo_2ano_rede["mat_2ano_biblioteca_sala_leitura"] = (
    df_censo_2ano_rede["QT_MAT_FUND_AI_2"]
    * df_censo_2ano_rede["IN_BIBLIOTECA_SALA_LEITURA"]
)

df_novos_indicadores = (
    df_censo_2ano_rede
    .groupby("CO_MUNICIPIO", as_index=False)
    .agg(
        mat_2ano_alimentacao=(
            "mat_2ano_alimentacao",
            "sum"
        ),
        mat_2ano_biblioteca_sala_leitura=(
            "mat_2ano_biblioteca_sala_leitura",
            "sum"
        )
    )
)

df_indicadores_censo = df_indicadores_censo.merge(
    df_novos_indicadores,
    on="CO_MUNICIPIO",
    how="left",
    validate="one_to_one"
)

df_indicadores_censo["prop_mat_2ano_alimentacao"] = (
    df_indicadores_censo["mat_2ano_alimentacao"]
    / df_indicadores_censo["total_mat_2ano"]
)

df_indicadores_censo["prop_mat_2ano_biblioteca_sala_leitura"] = (
    df_indicadores_censo["mat_2ano_biblioteca_sala_leitura"]
    / df_indicadores_censo["total_mat_2ano"]
)

df_indicadores_censo.head()

In [0]:
# Objetivo:
#
# Validar conjuntamente os três indicadores municipais
# construídos a partir do Censo Escolar 2024.
#
# Justificativa:
#
# Os indicadores representam proporções ponderadas pelas
# matrículas do 2º ano e, por definição, devem assumir
# valores entre 0 e 1.
#
# A metodologia utilizada foi previamente validada na
# construção do indicador de internet para aprendizagem
# e posteriormente aplicada aos indicadores de alimentação
# escolar e biblioteca e/ou sala de leitura.
#
# Antes de prosseguir para a integração com a base de
# alunos, é necessário verificar a completude e a
# consistência matemática dos três indicadores.
#
# Ação:
#
# Para cada indicador, verifica a quantidade de valores
# ausentes, os valores mínimo e máximo e a quantidade
# de registros fora do intervalo esperado [0, 1].

indicadores_censo = [
    "prop_mat_2ano_internet_aprendizagem",
    "prop_mat_2ano_alimentacao",
    "prop_mat_2ano_biblioteca_sala_leitura"
]

auditoria_indicadores_censo = pd.DataFrame({
    "ausentes": (
        df_indicadores_censo[indicadores_censo]
        .isna()
        .sum()
    ),
    "minimo": (
        df_indicadores_censo[indicadores_censo]
        .min()
    ),
    "maximo": (
        df_indicadores_censo[indicadores_censo]
        .max()
    ),
    "fora_intervalo": [
        (~df_indicadores_censo[coluna].between(0, 1)).sum()
        for coluna in indicadores_censo
    ]
})

auditoria_indicadores_censo

### Definição da tabela municipal de indicadores

Após a construção e validação dos indicadores, a tabela municipal
será reduzida às variáveis necessárias para sua posterior integração
com a base de alunos.

As colunas utilizadas como apoio aos cálculos — total de matrículas
do 2º ano e numeradores associados a cada característica — foram
mantidas durante a etapa de construção para permitir a auditoria
dos resultados, mas não serão incorporadas à base analítica final.

A tabela municipal manterá:

- `CO_MUNICIPIO`, como chave técnica de integração;
- `prop_mat_2ano_internet_aprendizagem`;
- `prop_mat_2ano_alimentacao`;
- `prop_mat_2ano_biblioteca_sala_leitura`.

Dessa forma, serão incorporadas à base de alunos somente as três
novas características contextuais derivadas do Censo Escolar 2024,
evitando a inclusão de variáveis intermediárias utilizadas
exclusivamente no processo de cálculo.

In [0]:
# Objetivo:
#
# Criar a tabela municipal de indicadores do
# Censo Escolar 2024 destinada à integração com
# a base de alunos.
#
# Justificativa:
#
# A tabela utilizada durante a construção dos indicadores
# contém variáveis auxiliares, como o total de matrículas
# do 2º ano e os numeradores utilizados nos cálculos.
#
# Essas variáveis foram importantes para a auditoria,
# mas não representam características que precisam ser
# incorporadas à base analítica de alunos.
#
# Para a integração, serão mantidas somente a chave
# municipal e as três proporções ponderadas construídas
# e validadas anteriormente.
#
# Ação:
#
# Seleciona a chave CO_MUNICIPIO e os três indicadores
# municipais e cria uma cópia independente da tabela
# utilizada durante as etapas de cálculo e auditoria.

df_censo_municipal = df_indicadores_censo[
    [
        "CO_MUNICIPIO",
        "prop_mat_2ano_internet_aprendizagem",
        "prop_mat_2ano_alimentacao",
        "prop_mat_2ano_biblioteca_sala_leitura"
    ]
].copy()

df_censo_municipal.shape

In [0]:
# Objetivo:
#
# Auditar a tabela municipal do Censo Escolar 2024
# antes de sua integração com a base de alunos + Atlas.
#
# Justificativa:
#
# A tabela df_censo_municipal representa o produto final
# da etapa de construção dos indicadores e contém somente
# a chave municipal e as três características que serão
# incorporadas à base analítica.
#
# Antes do merge, é necessário confirmar que a tabela
# preserva a granularidade de um registro por município,
# que a chave CO_MUNICIPIO permanece única e completa e
# que os indicadores não apresentam valores ausentes.
#
# Ação:
#
# Verifica a quantidade de registros, municípios
# distintos, duplicidades e valores ausentes na chave
# e nos três indicadores.

pd.Series({
    "registros": len(df_censo_municipal),
    "municipios_distintos": (
        df_censo_municipal["CO_MUNICIPIO"].nunique()
    ),
    "municipios_duplicados": (
        df_censo_municipal["CO_MUNICIPIO"].duplicated().sum()
    ),
    "chave_ausente": (
        df_censo_municipal["CO_MUNICIPIO"].isna().sum()
    ),
    "internet_ausente": (
        df_censo_municipal[
            "prop_mat_2ano_internet_aprendizagem"
        ].isna().sum()
    ),
    "alimentacao_ausente": (
        df_censo_municipal[
            "prop_mat_2ano_alimentacao"
        ].isna().sum()
    ),
    "biblioteca_ausente": (
        df_censo_municipal[
            "prop_mat_2ano_biblioteca_sala_leitura"
        ].isna().sum()
    )
})

In [0]:
# Objetivo:
#
# Verificar a compatibilidade estrutural das chaves
# municipais que serão utilizadas na integração entre
# a base de alunos + Atlas e os indicadores derivados
# do Censo Escolar 2024.
#
# Justificativa:
#
# A tabela municipal do Censo possui uma linha por
# município e utilizará CO_MUNICIPIO como chave de
# integração.
#
# Antes do merge, é necessário confirmar que a chave
# correspondente na base de alunos + Atlas possui tipo
# compatível e conhecer sua completude e cardinalidade.
#
# Essa verificação evita realizar conversões ou
# tratamentos sem evidência de necessidade.
#
# Ação:
#
# Compara o tipo das chaves municipais, contabiliza
# valores ausentes e verifica a quantidade de municípios
# distintos presentes em cada base.

pd.DataFrame({
    "base": [
        "alunos_atlas",
        "censo_municipal"
    ],
    "tipo": [
        df_alunos_atlas["CO_MUNICIPIO"].dtype,
        df_censo_municipal["CO_MUNICIPIO"].dtype
    ],
    "ausentes": [
        df_alunos_atlas["CO_MUNICIPIO"].isna().sum(),
        df_censo_municipal["CO_MUNICIPIO"].isna().sum()
    ],
    "municipios_distintos": [
        df_alunos_atlas["CO_MUNICIPIO"].nunique(),
        df_censo_municipal["CO_MUNICIPIO"].nunique()
    ]
})

In [0]:
# Objetivo:
#
# Verificar a cobertura dos municípios presentes na
# base de alunos + Atlas pelos indicadores municipais
# derivados do Censo Escolar 2024.
#
# Justificativa:
#
# A base de alunos + Atlas possui 5.556 municípios
# distintos com código informado, enquanto a tabela
# municipal do Censo contém 5.570 municípios.
#
# A diferença na quantidade de municípios não representa,
# isoladamente, um problema para a integração. Como o
# objetivo é incorporar os indicadores do Censo à base
# de alunos, é necessário verificar se todos os municípios
# presentes nos alunos possuem correspondência na tabela
# municipal do Censo.
#
# Os 510 registros sem CO_MUNICIPIO na base de alunos
# são preservados separadamente dessa comparação, pois
# não possuem chave disponível para correspondência.
#
# Ação:
#
# Compara os conjuntos de códigos municipais das duas
# bases e contabiliza os municípios presentes nos alunos
# sem correspondência no Censo e os municípios presentes
# no Censo que não aparecem na base de alunos.

municipios_alunos = set(
    df_alunos_atlas["CO_MUNICIPIO"]
    .dropna()
    .astype("int64")
)

municipios_censo = set(
    df_censo_municipal["CO_MUNICIPIO"]
)

municipios_alunos_sem_censo = (
    municipios_alunos - municipios_censo
)

municipios_censo_sem_alunos = (
    municipios_censo - municipios_alunos
)

pd.Series({
    "municipios_alunos": len(municipios_alunos),
    "municipios_censo": len(municipios_censo),
    "alunos_sem_censo": len(municipios_alunos_sem_censo),
    "censo_sem_alunos": len(municipios_censo_sem_alunos)
})

In [0]:
# Objetivo:
#
# Investigar o município presente na base de alunos
# + Atlas que não possui correspondência na tabela
# municipal derivada do Censo Escolar 2024.
#
# Justificativa:
#
# A auditoria de cobertura das chaves identificou um
# município presente na base de alunos e ausente na
# tabela municipal do Censo.
#
# Como essa ausência poderá resultar em valores nulos
# nos indicadores após a integração, é necessário
# identificar o município e dimensionar a quantidade
# de alunos potencialmente afetados antes de qualquer
# decisão de tratamento.
#
# Ação:
#
# Seleciona os registros da base de alunos + Atlas cujo
# código municipal pertence ao conjunto de municípios
# sem correspondência no Censo e apresenta sua
# identificação e quantidade de alunos.

(
    df_alunos_atlas[
        df_alunos_atlas["CO_MUNICIPIO"]
        .isin(municipios_alunos_sem_censo)
    ]
    .groupby(
        ["CO_MUNICIPIO", "NO_MUNICIPIO", "SG_UF"],
        dropna=False
    )
    .size()
    .reset_index(name="quantidade_alunos")
)

### Auditoria de correspondência das chaves municipais

Antes da integração dos indicadores do Censo Escolar 2024 com a
base de alunos + Atlas, foi realizada uma auditoria da correspondência
entre os códigos municipais presentes nas duas fontes.

A base de alunos + Atlas contém **5.556 municípios distintos** com
código informado, enquanto a tabela municipal construída a partir
do Censo Escolar 2024 contém **5.570 municípios**.

A comparação entre os conjuntos de códigos identificou:

- **1 município presente na base de alunos e ausente na tabela do Censo**;
- **15 municípios presentes na tabela do Censo e ausentes na base de alunos**.

O município sem correspondência no Censo foi identificado como
**Boa Esperança do Norte (MT)**, código `5101837`, associado a
**110 alunos** na base analisada.

A ausência de correspondência é compatível com a diferença temporal
entre as fontes. Boa Esperança do Norte é um município de criação
recente, instalado oficialmente em **1º de janeiro de 2025**, enquanto
os indicadores municipais utilizados neste projeto foram construídos
a partir do **Censo Escolar 2024**.

Dessa forma, a ausência desse código na agregação do Censo Escolar
2024 não será tratada como erro de chave nem corrigida automaticamente.
Os 110 alunos permanecerão na base, mas não receberão os indicadores
municipais derivados do Censo Escolar 2024 por não existir
correspondência direta para seu município nessa referência temporal.

Além desses registros, a base de alunos possui **510 alunos sem
CO_MUNICIPIO informado**, que também não poderão receber os indicadores
por meio da chave municipal.

Essas duas situações possuem origens distintas e serão preservadas
separadamente na interpretação das ausências após a integração:

- **510 alunos:** ausência da chave municipal na base de origem;
- **110 alunos:** município identificado, porém sem correspondência
  direta no Censo Escolar 2024 devido à diferença temporal entre
  as fontes.

Assim, após a integração, espera-se que até **620 registros** não
possuam valores para os três indicadores derivados do Censo Escolar,
sem que essas ausências sejam automaticamente interpretadas como
falhas no processo de merge.

## 4. Integração dos indicadores do Censo Escolar à base de alunos

Após a construção e validação dos indicadores municipais derivados
do Censo Escolar 2024, a próxima etapa consiste em incorporá-los à
base de alunos já enriquecida com os indicadores socioeconômicos
do Atlas do Desenvolvimento Humano.

A tabela `df_censo_municipal` possui granularidade municipal, com
uma linha por município, e contém três indicadores:

- `prop_mat_2ano_internet_aprendizagem`;
- `prop_mat_2ano_alimentacao`;
- `prop_mat_2ano_biblioteca_sala_leitura`.

Antes da integração, a chave `CO_MUNICIPIO` foi auditada nas duas
bases. A tabela municipal do Censo possui 5.570 códigos municipais
distintos, sem duplicidades ou valores ausentes, enquanto a base de
alunos + Atlas possui 5.556 municípios distintos com código informado.

A auditoria de correspondência identificou um município presente na
base de alunos sem correspondência direta na agregação construída a
partir do Censo Escolar 2024: Boa Esperança do Norte (MT), código
`5101837`, associado a 110 alunos.

Além disso, a base de alunos possui 510 registros sem `CO_MUNICIPIO`
informado. Essas situações foram previamente documentadas e serão
preservadas durante a integração.

Como vários alunos podem pertencer ao mesmo município e a tabela do
Censo possui apenas um registro por município, a relação esperada
entre as bases é muitos-para-um (`many_to_one`).

A integração será realizada por meio de um `left merge`, utilizando
`CO_MUNICIPIO` como chave, de modo a preservar todos os registros da
base de alunos.

Após o merge, serão auditadas a quantidade de registros, a
cardinalidade da integração e a completude dos três novos indicadores,
com atenção especial às ausências previamente identificadas.

In [0]:
# Objetivo:
#
# Integrar os indicadores municipais derivados do
# Censo Escolar 2024 à base de alunos + Atlas.
#
# Justificativa:
#
# A tabela df_censo_municipal possui uma linha por
# município e contém três indicadores municipais
# construídos e validados anteriormente.
#
# Como vários alunos podem pertencer ao mesmo município,
# a relação esperada entre a base de alunos + Atlas e a
# tabela municipal do Censo é muitos-para-um.
#
# O left merge é utilizado para preservar todos os
# registros da base de alunos, inclusive aqueles sem
# CO_MUNICIPIO informado ou sem correspondência direta
# no Censo Escolar 2024.
#
# Ação:
#
# Realiza a integração utilizando CO_MUNICIPIO como
# chave, valida a cardinalidade many_to_one e adiciona
# uma coluna auxiliar para posterior auditoria da
# correspondência entre as bases.

df_alunos_atlas_censo = df_alunos_atlas.merge(
    df_censo_municipal,
    on="CO_MUNICIPIO",
    how="left",
    validate="many_to_one",
    indicator=True
)

df_alunos_atlas_censo.shape

In [0]:
# Objetivo:
#
# Auditar a correspondência entre a base de alunos
# + Atlas e os indicadores municipais derivados do
# Censo Escolar 2024.
#
# Justificativa:
#
# O merge foi realizado com indicator=True, criando
# a coluna auxiliar _merge, que permite identificar
# quais registros encontraram correspondência na tabela
# municipal do Censo.
#
# A auditoria prévia das chaves indicou duas situações
# que podem resultar em ausência de correspondência:
#
# - 510 alunos sem CO_MUNICIPIO informado;
# - 110 alunos de Boa Esperança do Norte (MT), município
#   sem correspondência direta no Censo Escolar 2024.
#
# Antes de analisar os valores ausentes nos indicadores,
# é necessário verificar se o resultado do merge é
# compatível com essas expectativas.
#
# Ação:
#
# Contabiliza os registros segundo a situação de
# correspondência registrada na coluna _merge.

df_alunos_atlas_censo["_merge"].value_counts()

In [0]:
# Objetivo:
#
# Reconciliar os registros sem correspondência no
# Censo Escolar 2024 segundo as causas identificadas
# antes da integração.
#
# Justificativa:
#
# A auditoria do merge identificou 620 registros
# classificados como left_only.
#
# Antes da integração, haviam sido identificados:
#
# - 510 alunos sem CO_MUNICIPIO informado;
# - 110 alunos de Boa Esperança do Norte (MT),
#   município sem correspondência direta na tabela
#   derivada do Censo Escolar 2024.
#
# A reconciliação permite confirmar que todos os casos
# sem correspondência são explicados exclusivamente
# por essas situações previamente conhecidas.
#
# Ação:
#
# Contabiliza separadamente os registros left_only
# sem código municipal e os pertencentes a
# Boa Esperança do Norte.

left_only = df_alunos_atlas_censo[
    df_alunos_atlas_censo["_merge"] == "left_only"
]

pd.Series({
    "sem_codigo_municipal": (
        left_only["CO_MUNICIPIO"].isna().sum()
    ),
    "boa_esperanca_do_norte": (
        left_only["CO_MUNICIPIO"].eq(5101837).sum()
    ),
    "outros_casos": (
        (
            left_only["CO_MUNICIPIO"].notna()
            & ~left_only["CO_MUNICIPIO"].eq(5101837)
        ).sum()
    )
})

In [0]:
# Objetivo:
#
# Validar a completude dos indicadores derivados do
# Censo Escolar 2024 após sua integração à base de
# alunos + Atlas.
#
# Justificativa:
#
# A auditoria do merge identificou 620 registros sem
# correspondência na tabela municipal do Censo:
#
# - 510 alunos sem CO_MUNICIPIO informado;
# - 110 alunos de Boa Esperança do Norte (MT).
#
# Como os três indicadores foram incorporados por meio
# da mesma chave municipal, espera-se que cada um apresente
# valores ausentes exatamente nesses 620 registros.
#
# A comparação entre o número observado e o esperado
# permite verificar se surgiram ausências adicionais
# durante a integração.
#
# Ação:
#
# Contabiliza os valores ausentes nos três indicadores
# e compara cada resultado com o total esperado de 620.

indicadores_censo = [
    "prop_mat_2ano_internet_aprendizagem",
    "prop_mat_2ano_alimentacao",
    "prop_mat_2ano_biblioteca_sala_leitura"
]

esperado_ausentes = 620

pd.DataFrame({
    "ausentes": (
        df_alunos_atlas_censo[indicadores_censo]
        .isna()
        .sum()
    ),
    "esperado": esperado_ausentes
}).assign(
    conforme=lambda x: x["ausentes"] == x["esperado"]
)

In [0]:
# Objetivo:
#
# Verificar se a integração com os indicadores do
# Censo Escolar 2024 preservou a granularidade da
# base de alunos.
#
# Justificativa:
#
# O merge foi realizado com cardinalidade many_to_one
# e preservou a quantidade total de registros da base
# original.
#
# Antes de concluir a integração, é necessário confirmar
# que cada ID_ALUNO continua aparecendo uma única vez,
# garantindo que a incorporação dos indicadores municipais
# não tenha introduzido duplicidades.
#
# Ação:
#
# Compara a quantidade total de registros com o número
# de alunos distintos e contabiliza eventuais
# duplicidades em ID_ALUNO.

pd.Series({
    "registros": len(df_alunos_atlas_censo),
    "alunos_distintos": (
        df_alunos_atlas_censo["ID_ALUNO"].nunique()
    ),
    "alunos_duplicados": (
        df_alunos_atlas_censo["ID_ALUNO"].duplicated().sum()
    )
})

### Resultado da integração com o Censo Escolar 2024

A integração dos indicadores municipais derivados do Censo Escolar
2024 com a base de alunos + Atlas foi realizada por meio de um
`left merge`, utilizando `CO_MUNICIPIO` como chave e validando a
cardinalidade `many_to_one`.

A operação preservou integralmente os **1.966.605 registros** da base
de alunos e manteve sua granularidade original, com **1.966.605 alunos
distintos** e nenhuma duplicidade em `ID_ALUNO`.

A auditoria da correspondência registrou:

- **1.965.985 registros** com correspondência (`both`);
- **620 registros** sem correspondência (`left_only`);
- nenhum registro `right_only`.

Os 620 casos sem correspondência foram integralmente reconciliados
com as situações identificadas antes do merge:

- **510 alunos** sem `CO_MUNICIPIO` informado;
- **110 alunos** de Boa Esperança do Norte (MT), código `5101837`,
  sem correspondência direta na agregação construída a partir do
  Censo Escolar 2024;
- **0 casos adicionais** sem explicação.

Os três indicadores incorporados apresentaram exatamente **620 valores
ausentes cada**, correspondendo aos mesmos registros sem possibilidade
de associação municipal. Não foram identificadas ausências adicionais
produzidas pelo processo de integração.

Dessa forma, a integração preservou a população e a granularidade da
base de alunos, e todas as ausências introduzidas nos indicadores do
Censo Escolar possuem origem conhecida e documentada.

In [0]:
# Objetivo:
#
# Remover a coluna auxiliar utilizada na auditoria
# da integração com o Censo Escolar 2024.
#
# Justificativa:
#
# A coluna _merge foi criada temporariamente pelo
# parâmetro indicator=True para permitir a identificação
# dos registros com e sem correspondência entre as bases.
#
# Após a reconciliação dos 620 registros classificados
# como left_only e a validação da completude dos novos
# indicadores, essa coluna já cumpriu sua finalidade
# de auditoria e não representa uma característica
# analítica dos alunos.
#
# Ação:
#
# Remove a coluna auxiliar _merge e verifica as
# dimensões da base resultante.

df_alunos_atlas_censo = df_alunos_atlas_censo.drop(
    columns="_merge"
)

df_alunos_atlas_censo.shape

In [0]:
# Objetivo:
#
# Inspecionar os indicadores do Censo Escolar 2024
# após sua incorporação à base analítica de alunos.
#
# Justificativa:
#
# As etapas anteriores confirmaram a preservação da
# quantidade de registros, da granularidade dos alunos
# e a origem das ausências nos novos indicadores.
#
# Antes da persistência da base resultante, uma inspeção
# visual permite verificar as novas características no
# contexto dos registros individuais.
#
# Ação:
#
# Exibe uma amostra das colunas de identificação
# municipal e dos três indicadores incorporados.

df_alunos_atlas_censo[
    [
        "ID_ALUNO",
        "CO_MUNICIPIO",
        "NO_MUNICIPIO",
        "SG_UF",
        "prop_mat_2ano_internet_aprendizagem",
        "prop_mat_2ano_alimentacao",
        "prop_mat_2ano_biblioteca_sala_leitura"
    ]
].head(10)

### Base analítica após a integração

Após a incorporação dos indicadores derivados do Censo Escolar 2024,
a base analítica passou a conter **1.966.605 registros e 36 colunas**,
preservando a granularidade original de uma linha por aluno.

Foram adicionadas três características contextuais municipais:

- `prop_mat_2ano_internet_aprendizagem`;
- `prop_mat_2ano_alimentacao`;
- `prop_mat_2ano_biblioteca_sala_leitura`.

Como os indicadores possuem granularidade municipal, alunos
pertencentes ao mesmo município recebem os mesmos valores. Essa
repetição é esperada e representa a associação de características
do contexto educacional municipal aos registros individuais dos
alunos.

As três novas variáveis devem ser interpretadas como proporções das
matrículas do 2º ano associadas a estabelecimentos que apresentam
cada característica, e não como medidas individuais de acesso ou
oferta para cada aluno.

A integração preservou todos os registros da base original. Os
valores ausentes existentes nos novos indicadores estão restritos
aos 620 registros previamente identificados e documentados durante
a auditoria das chaves.

## 5. Persistência da base enriquecida

Após a construção dos indicadores municipais do Censo Escolar 2024
e sua integração à base de alunos + Atlas, esta etapa tem como objetivo
persistir o novo produto analítico na camada Gold.

A base resultante, denominada `alunos_atlas_censo`, preserva a
granularidade de uma linha por aluno e contém **1.966.605 registros
e 36 colunas**.

Em relação à versão anterior `alunos_atlas`, foram incorporadas três
novas características contextuais municipais:

- `prop_mat_2ano_internet_aprendizagem`;
- `prop_mat_2ano_alimentacao`;
- `prop_mat_2ano_biblioteca_sala_leitura`.

A persistência será realizada em um novo diretório, preservando os
produtos anteriores e tornando explícita a linhagem de enriquecimento
da base:

A base será salva utilizando separador `;`, codificação `utf-8-sig`
e sem a inclusão do índice do DataFrame.

Após a gravação, o arquivo será novamente carregado e auditado para
confirmar que suas dimensões e características essenciais foram
preservadas durante o processo de persistência.

In [0]:
# Objetivo:
#
# Persistir a base analítica enriquecida com os
# indicadores municipais derivados do Censo Escolar 2024.
#
# Justificativa:
#
# Após a integração e auditoria dos três novos indicadores,
# a base df_alunos_atlas_censo preservou os 1.966.605
# registros de alunos e passou a conter 36 colunas.
#
# O novo produto será armazenado separadamente na camada
# Gold, preservando as versões anteriores e tornando
# explícita a linhagem de enriquecimento da base.
#
# Ação:
#
# Salva a base alunos + Atlas + Censo Escolar em arquivo
# CSV, utilizando o padrão de separador e codificação
# adotado nos demais produtos da camada Gold.

df_alunos_atlas_censo.to_csv(
    "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/gold/alunos_atlas_censo/alunos_atlas_censo.csv",
    sep=";",
    encoding="utf-8-sig",
    index=False
)

In [0]:
# Objetivo:
#
# Validar o arquivo persistido após a
# integração com o Censo Escolar 2024.
#
# Justificativa:
#
# A releitura do arquivo permite confirmar
# que a base foi gravada corretamente e
# preservou sua dimensão antes de ser utilizada
# nas próximas etapas do projeto.
#
# Ação:
#
# Realiza uma leitura de controle do arquivo
# persistido e verifica sua dimensão.

df_validacao = pd.read_csv(
    "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/gold/alunos_atlas_censo/alunos_atlas_censo.csv",
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

df_validacao.shape